# Fetching data

Based on: https://github.com/ageron/handson-ml2

In [ ]:
import os
import tarfile
from six.moves import urllib
import pandas as pd

DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
HOUSING_PATH = os.path.join("datasets", "housing")
HOUSING_URL = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"

def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    if not os.path.isdir(housing_path):
        os.makedirs(housing_path)
        tgz_path = os.path.join(housing_path, "housing.tgz")
        urllib.request.urlretrieve(housing_url, tgz_path)
        housing_tgz = tarfile.open(tgz_path)
        housing_tgz.extractall(path=housing_path)
        housing_tgz.close()

fetch_housing_data()
housing = load_housing_data()
print(housing.info())
housing.head()

# Train/Test Split

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
housing["income_cat"] = pd.cut(housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
    )
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing = pd.DataFrame(strat_train_set) # type: ignore
test_set = pd.DataFrame(strat_test_set) # type: ignore
print(f"Train set size: {len(housing)}")
print(f"Test set size: {len(test_set)}")
housing.plot(kind="scatter", x="longitude", y="latitude")
test_set.plot(kind="scatter", x="longitude", y="latitude")

In [ ]:
housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]
housing["population_per_household"]=housing["population"]/housing["households"]

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

## Preencher valores vazios

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
housing_num = housing.drop("ocean_proximity", axis=1) # type: ignore
imputer.fit(housing_num) # type: ignore
print(imputer.statistics_)
print(housing_num.median().values) # type: ignore

if type(housing) == pd.DataFrame: # type: ignore
    X = imputer.transform(housing_num) # type: ignore
    housing_tr = pd.DataFrame(X, columns=housing_num.columns) # type: ignore

## Ordinal Encoder

Torna uma variável categórica para um codigo numérico
(como categoria 0, 1, 2 ...)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder # type: ignore

ordinal_encoder = OrdinalEncoder()
housing_cat = housing[["ocean_proximity"]] # type: ignore
ordinal_encoder.fit(housing_cat) # type: ignore
housing_cat_encoded = ordinal_encoder.transform(housing_cat) # type: ignore
print(housing_cat[:10])
print(housing_cat_encoded[:10])
ordinal_encoder.categories_

## One Hot Encoder

Melhor quando uma variável categórica não for ordinal (isto é, a categoria 2 não é maior do que a 1)

In [ ]:
from sklearn.preprocessing import OneHotEncoder # type: ignore
onehot_encoder = OneHotEncoder()
onehot_encoder.fit(housing_cat) # type: ignore
housing_cat_1hot = onehot_encoder.transform(housing_cat) # type: ignore
print(onehot_encoder.categories_)
print(housing_cat[:10])
print(housing_cat_1hot[:10])
housing_cat_1hot.toarray()[:10] # type: ignore

## Combinação de Atributos

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6
class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True): # no *args or **kargs
        self.add_bedrooms_per_room = add_bedrooms_per_room

    def fit(self, X, y=None):
        return self # nothing else to do

    def transform(self, X, y=None):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household,
            bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
housing_extra_attribs = attr_adder.transform(housing.values) # type: ignore

# Pipeline de Formatação

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('attribs_adder', CombinedAttributesAdder()),
    ('std_scaler', StandardScaler()),
])
housing_num_tr = num_pipeline.fit_transform(housing_num) # type: ignore
housing_num_tr

In [ ]:
from sklearn.compose import ColumnTransformer
num_attribs = list(housing_num) # type: ignore
cat_attribs = ["ocean_proximity"]
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs),
])
housing_prepared = full_pipeline.fit_transform(housing) # type: ignore
housing_prepared

# Regressão Linear

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels) # type: ignore


some_data = housing.iloc[:5] # type: ignore
some_labels = housing_labels.iloc[:5] # type: ignore
housing_predictions = lin_reg.predict(housing_prepared) # type: ignore
lin_mse = mean_squared_error(housing_labels, housing_predictions)
lin_rmse = np.sqrt(lin_mse)
print("Predictions:", housing_predictions[:5])
print("Labels:", list(some_labels))

print("Root Mean Squared Error:", lin_rmse)

# Árvore de Decisão